# Fine-tune Laya on NBA possessions (Colab, 1 GPU)

Runtime -> Change runtime type -> **A100** if you have compute units, otherwise **L4**. T4 works but is the slowest.

Inputs: `train.jsonl`, `calib.jsonl`, `test.jsonl` from `make export` in the NBA_Laya repo. Upload them to Drive under `NBA_Laya/finetune/data/`, or drop them into `/content/data/` by hand.

Output: a Laya checkpoint under `laya_nba/` that `laya-serve` can load, plus `test_scoreboard.json`.

In [ ]:
!pip -q install laya safetensors
import os; os.environ['HF_HUB_DISABLE_XET'] = '1'

In [ ]:
# Option A: Google Drive
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/NBA_Laya/finetune/data'
OUT  = '/content/drive/MyDrive/NBA_Laya/laya_nba'
# Option B: uploaded to /content/data instead
# DATA, OUT = '/content/data', '/content/laya_nba'
!ls -la $DATA && wc -l $DATA/*.jsonl

In [ ]:
# The training script from the repo (finetune/train.py). Pull it fresh.
!curl -fsSL -o train.py https://raw.githubusercontent.com/anovick1/HoopsOracle/main/finetune/train.py
# Or upload train.py to /content by hand and skip the curl.

## First pass: 100k items, 2 epochs
A full season is about 780k items. Start at 100k. The log prints elapsed seconds every 100 steps, so after a few minutes you can project the full run. Test and calib are capped at 20k items each (`--max-eval`), still from held-out games.

In [ ]:
!python train.py --data $DATA --out $OUT --epochs 2 --max-train 100000 --micro-batch 16 --grad-accum 4

## Full run
`--max-train 0` uses every training item. Only worth it if the first pass beat the zero-shot line. The rolling checkpoint under `OUT/checkpoint_latest` is saved after each epoch, so a disconnect loses at most one epoch.

In [ ]:
# !python train.py --data $DATA --out $OUT --epochs 3 --max-train 0 --micro-batch 16 --grad-accum 4

## Use it
Download `OUT/` to your machine and serve it:

```bash
LAYA_MODELS=  laya-serve   # then point the client at the folder, or:
python -c "import laya; a = laya.load('laya_nba'); print(a.predict(state, questions))"
```

Then `nba-laya replay --model laya` on the held-out games and `nba-laya scoreboard` next to the rules baseline.